In [1]:
import numpy as np
import pandas as pd
import random
import itertools
SEED = 100
random.seed(SEED) #For reproducibility (although datamimic_ce has its own random generator / seed)
rng = np.random.default_rng(seed=SEED)

In [2]:
from types import CellType
from enum import StrEnum
class PaymentActions(StrEnum):
    PAYMENT_EXPRESS = "Payment_Express"
    SERVICE_PAYMENT = "Service_payment"
    #CREDITCARD_PAYMENT = "CreditCard_payment"
    #INTERBANK_PAYMENT = "Interbank_payment"
    #BENEF_PAYMENT = "Benef_payment"
    BENEF_INTERBANK_PAYMENT = "Benef_interbank_payment"
    #DEPOSIT = "Deposit"

class UserActions(StrEnum):
    DASHBOARD = "Dashboard"
    ACCOUNT_DETAILS = "Account_Details"
    TRANS_HISTORICAL = "Transaction_Historical"

class AccountTypes(StrEnum):
    CHECKING = "CHECKING"
    SAVINGS = "SAVINGS"
    MONEYMARKET = "MONEY_MARKET",
    CERTIFICATE_OF_DEPOSIT = "CERTIFICATE_OF_DEPOSIT"

class AccountBalanceStatus(StrEnum):
    NEGATIVE = "NEGATIVE"
    POSITIVE = "POSITIVE"
    FLAT = "FLAT"


destiny_accounts = {
    "SERVICE_PAYMENT": ["PHONE", "NETFLIX", "ELECTRICITY", "INTERNET", "WATER"],
    "PAYMENT_EXPRESS":["PURCHASES"],
    "BENEF_INTERBANK_PAYMENT": ["RENT", "FINANCIAMIENTO"],
}

destiny_accounts_amounts = {
    "SERVICE_PAYMENT": {
        "PHONE": {"mean": 1500.00, "deviation": 20.00},
        "NETFLIX": {"mean": 600.00, "deviation": 0.00},
        "ELECTRICITY": {"mean": 3000.00, "deviation": 1000.00},
        "INTERNET": {"mean": 2400.00, "deviation": 0.00},
        "WATER": {"mean": 300.00, "deviation": 100.00}
    },
    "PAYMENT_EXPRESS": {
        "PURCHASES": {"mean": 5500.00, "deviation": 2000.00}
    },
    "BENEF_INTERBANK_PAYMENT": {
        "RENT": {"mean": 10000.00, "deviation": 0.00},
        "FINANCIAMIENTO": {"mean": 8000.00, "deviation":300.00}
    }
}

destiny_amounts_express = {
    "PURCHASES": 3500.00
}

destiny_accounts_interbank = {
    "RENT": 10000.00,
    "FINANCIAMIENTO": 10000.00
}

In [3]:
destiny_accounts_amounts["SERVICE_PAYMENT"]["ELECTRICITY"]

{'mean': 3000.0, 'deviation': 1000.0}

In [4]:
list(destiny_accounts.keys())

['SERVICE_PAYMENT', 'PAYMENT_EXPRESS', 'BENEF_INTERBANK_PAYMENT']

In [5]:
destiny_accounts.keys()
random.choice(list(destiny_accounts.keys()))

'SERVICE_PAYMENT'

In [6]:
!pip install datamimic-ce

In [7]:
from datamimic_ce.domains.common.literal_generators.datetime_generator import DateTimeGenerator
from datetime import datetime
from datamimic_ce.domains.finance.models.credit_card import BankAccount
from datamimic_ce.domains.finance.services.bank_service import BankService

In [8]:
#from datamimic_ce.domains.finance.services import BankAccountService

In [9]:
'''
class accountTypes(StrEnum):
    CHECKING = "CHECKING"
    SAVINGS = "SAVINGS"
    CREDITCARD = "CREDIT"
    MONEYMARKET = "MONEY_MARKET"
'''

accountTypes = ["CHECKING", "SAVINGS", "CREDIT", "MONEYMARKET", "CERTIFICATE_OF_DEPOSIT"]



In [10]:
from datamimic_ce.domains.finance.services import BankAccountService
from datamimic_ce.domains.finance.services import CreditCardService

#Generate a list of BankAccount entities
global accounts
accounts = {
    "CHECKING": [],
    "SAVINGS": [],
    "CERTIFICATE_OF_DEPOSIT": [],
    "MONEYMARKET": [],
}
def generateAccounts():
    #TODO: research how to implicitly tell generate_batch which type of account to generate
    batch_accounts = BankAccountService().generate_batch(50)
    #filtered_accounts = []
    account_types = accountTypes.copy()

    #Take only set of accounts with desired typologies
    for account in batch_accounts:
        #print("[INFO] Skipping account of type:", account.account_type)
        #if sum(len(val) for val in accounts.values()) >= number_of_accounts:
        #    break
        if account.account_type == AccountTypes.SAVINGS and accounts["SAVINGS"]==[]:
            accounts["SAVINGS"].append(account)
        elif account.account_type == AccountTypes.CHECKING and accounts["CHECKING"]==[]:
            accounts["CHECKING"].append(account)
        elif account.account_type == AccountTypes.CERTIFICATE_OF_DEPOSIT and accounts["CERTIFICATE_OF_DEPOSIT"]==[]:
            accounts["CERTIFICATE_OF_DEPOSIT"].append(account)
        elif account.account_type == AccountTypes.MONEYMARKET and accounts["MONEYMARKET"]==[]:
            accounts["MONEYMARKET"].append(account)
        else:
            #print("[INFO] Skipping account of type:", account.account_type)
            continue
            #account_types.remove(account.account_type)

generateAccounts()
#generate synthetic credit card entity
credit_card = CreditCardService().generate()

#Create an array of at the most 4 accounts with unique account types
#account_array = np.array(filtered_accounts)
#account_array = account_array[np.unique_values]

In [11]:
accounts["SAVINGS"][0].account_number

'CDJQ12827584149050'

In [13]:
#foo = updateBalance(AccountTypes.SAVINGS, -200)
#print(type(foo))

In [14]:
import copy
import uuid

user_income = 25000.00

#Helper function to obtain a True value with a p probability
def chance(p = 0.5):
    return rng.choice([True, False], p=[p, 1-p])

#Obtain a payment amount within the [mean-deviation, mean+deviation]
def amountByChance(mean, deviation):
    sign = rng.choice([-1, 1])
    if deviation == 0:
        return mean
    return mean+sign*rng.choice(int(deviation))

#Generate a sessionId to attach a set of actions to that session
def generateSessionId():
    return str(uuid.uuid4())

#Get total sum of available balances for all accounts
def getBalances():
    total = 0
    #Get total balance for SAVING, CHECKING and CERTIFICATE_OF_DEPOSIT
    for account in accounts.values():
        if account != []:
            total += account[0].balance
    #Substract any pending balance from credit card entity
    total -= credit_card.current_balance
    return total

# Update the balance for an account of type account_type in the user's portfolio
def updateBalance(account_type=AccountTypes, amount=0):
    balance = 0
    if accounts[account_type] != []:
        accounts[account_type][0].balance += amount
        balance = accounts[account_type][0].balance
    return balance

#Randomly pick a wage payment day with equaly distributed probabilities
def getPayday(p = [0.25, 0.25, 0.25, 0.25], days=['13', '14', '15', '16']):
    return rng.choice(days, p=p)



In [15]:
# Returns a list of number_of_dates dates randomly generated for the year and month given
def generateDates(number_of_dates=250, year=2025, month=1):
    # Determine the last_day in the month to avoid ValueError scenarios with DateTimeGenerator
    match month:
        case 1 | 3 | 5 | 7 | 8 | 10 | 12:
            last_day = 31
        case 4 | 6 | 9 | 11:
            last_day = 30
        case 2:
            last_day = 28
    # Deterministic dates within the min-max period.
    gen = DateTimeGenerator(
        min=f'{year}-{month}-01 00:00:00', max=f'{year}-{month}-{last_day} 23:59:59', dom='1-4,14-17,27-30', random=True,
        #weekdays='Weekend',
        seed=SEED,
    )
    dates = [gen.generate() for i in range(number_of_dates)]
    dates.sort()
    return dates

In [16]:
#Define wage payment variables
first_half_paydays = [13, 14, 15, 16]
second_half_paydays = [27, 28, 29, 30]
first_half_pay = 25000
second_half_pay = 10000

#Keeps a registry of those days were payments were made for a given month
payment_track = []

#Helper function that returns a randomly picked payment type, payment action and due payment amount. The conditional
#probability for the action depends on the current day of the week, if wage payment
#was already done for the biweekly period and the quantity of payments already processed
def getNextPaymentAction(pending_payment_actions, dow=1, wage_received=False):
    # Damp the probability of a payment if the user has not received any payment just yet
    # to emulate a more likely payment of services and debt after receiving the salary
    global payment_track
    if not wage_received and chance(p=0.90): #With p certainty (if no salary payment recieved) no next payment action is done
        print("[INFO] returning empty due to wage not received")
        return ("", 0.00)
    # Increase likelyness of next payment action if dow is within the biweekly window
    # TODO calculate prob of payment happening according to how close it is to the mean of the paydays list (z-score)
    if dow not in first_half_paydays+second_half_paydays and chance(p=0.90):
        print("[INFO] returning empty due to dow not in window")
        return ("", 0.00)

    # Reduce the probability of performing all payments in the same day
    payments_count = payment_track.count(dow)
    #Avoid dividing by zero
    if payments_count > 1:
        recurrent_payment_probability = (1/payments_count)
        if chance(p=recurrent_payment_probability):
            print("[INFO] returning empty due to high number of payments already done on same day")
            return ("", 0.00)
    # Obtain pending payment types
    try:
        payments = list(pending_payment_actions.keys())
        if (payments==[]):
            payment_track = []
            print("[INFO] returning empty due to no pending payments")
            return ("", 0.00)

        if len(payments) >= 1:
            probabilities = [1/len(payments)]*len(payments)
            print(probabilities)
            payment_type = rng.choice(payments, p=probabilities)
            print(payment_type)
            #From the picked payment action type, choose also a subtype
            next_possible_actions = pending_payment_actions[payment_type]
            probabilities = [1/len(next_possible_actions)]*len(next_possible_actions)
            next_action = rng.choice(next_possible_actions, p=probabilities)
            print(next_action)
            #print(payments)
            #Get due payment
            payment_configuration = destiny_accounts_amounts[payment_type][next_action]
            due_amount = amountByChance(mean=payment_configuration["mean"], deviation=payment_configuration["deviation"])
            print(f"[INFO] due amount: {due_amount}")
            #global payment_track
            payment_track.append(dow)
            return (payment_type+"-"+next_action, due_amount)
    except AttributeError:
        print(f"[INFO] No pending payment actions were able to be extracted")
        return ("", 0.00)

In [17]:
# Generate number_of_dates within the min-max window. Each date will be related
#to an specific user action (transactional or not)

#TODO: handle all list datasets more efficiently in one common DataFrame
dates = []
user_actions = []
session_ids = []
balance = []

#Number of years and months to span
YEARS_SPAN = 2
years = [2024+year for year in range(YEARS_SPAN)]
months = [month for month in range(1, 13)]


pending_actions = copy.deepcopy(destiny_accounts)

dates = []#generateDates()

#dates are unsorted
#dates.sort()

#Initialize user_actions list with DASHBOARD consult actions
user_actions = []#[UserActions.DASHBOARD]*len(dates)

#Initialize session_ids list with empty strings
session_ids = []#[generateSessionId() for n in range(len(dates))]

#Initialize balance list with the total at epoch 0
balances = []#[getBalances()]*len(dates)

#The registry of the source accounts for the payments
accounts_from = []

#The historical track for the main account balance
main_savings_balance = []

#The historical track for the main account balance
main_checking_balance = []

#The balance status for the main savings account
main_savings_balance_status = []

#The balance status for the main checking account
main_checking_balance_status = []

#

#Number of years to span
#YEARS = 1
payment_flags = []

def generateUserActions():
    #destiny = destiny_accounts.copy()

    #generate monthly set of dates and transactions for given list of years
    for year, month in list(itertools.product(years, months)):
        print(f"[INFO] iterating for year {year} and month {month}")
        #Recreate pending_actions as all payments are due again for the current month
        pending_actions = copy.deepcopy(destiny_accounts)
        #current_year = f'202{year}'
        monthly_dates = generateDates(year=year, month=month)
        #print(f"[INFO] monthly dates: {len(monthly_dates)} {monthly_dates[-1]}")
        monthly_user_actions = [UserActions.DASHBOARD]*len(monthly_dates)
        monthly_session_ids = [generateSessionId() for n in range(len(monthly_dates))]
        #print(dates)
        daily_balances = [getBalances()]*len(monthly_dates)
        payment_accounts_from = [""]*len(monthly_dates)
        main_account_balances = [0]*len(monthly_dates)
        main_savings_status = [AccountBalanceStatus.FLAT]*len(monthly_dates)

        payment_due = [False]*len(monthly_dates)
        payday1 = getPayday(days=first_half_paydays)
        payday2 = getPayday(days=second_half_paydays)
        print(f"[INFO] Paydays: {payday1} and {payday2}")
        first_payment_received = False
        second_payment_received = False
        i = 0
        #for i in range(1, len(monthly_dates)):
        for date in monthly_dates:
            #print(f"[INFO] date: {date}")
            #Generate a session date and its sessinId

            #print("[INFO] original date: ", date)
            #Append Dashboard consult action
            #dates.append(date)
            #user_actions.append(UserActions.DASHBOARD)
            #TODO:
            #-Encapsulate all the following logic in a function,
            #-evaluate increment in seconds
            #date = monthly_dates[i]
            user_action = monthly_user_actions[i]
            session_id = monthly_session_ids[i]
            print("[INFO] date:", date)
            #print("[INFO] first payment date:", payday1)

            print("[INFO] user_action:", user_action)

            #Emulate a wage payment to the users SAVINGS account
            if date.day >= int(payday1) and not first_payment_received:
                print(f"[INFO] Registering first wage payment of the month on day: {date.day}")
                balance = updateBalance(AccountTypes.SAVINGS, first_half_pay)
                print(f"[INFO] Updated SAVINGS balance: {balance}")
                first_payment_received = True
                #payment_due[i] = True
            #elif date.day >= int(payday2) and not second_payment_received:

            #Obtain accounts balances previous to any transaction on the session
            total_balace = getBalances()
            current_savings_balance = accounts["SAVINGS"][0].balance
            previous_savings_balance = main_account_balances[i-1]
            if previous_savings_balance < current_savings_balance:
                account_status = AccountBalanceStatus.POSITIVE
            elif previous_savings_balance == current_savings_balance:
                account_status = AccountBalanceStatus.FLAT
            else:
                account_status = AccountBalanceStatus.NEGATIVE
            #Update balances lists for current epoch
            daily_balances[i] = total_balace
            main_account_balances[i] = current_savings_balance
            main_savings_status[i] = account_status
            payment_accounts_from[i] = ""
            #print("[INFO] total_balaces:", total_balaces)

            #Insert AccountDetails action within the next random number of minutes with p probability
            if chance(p=0.55) == True and user_action == UserActions.DASHBOARD:
              print(f"[INFO] changing to acc-det user_action")
              next_minute = random.randint(date.minute, 59)
              date = date.replace(minute=next_minute)
              #print(f"[INFO] date after summing {next_minute} mins:", date)
              monthly_dates.insert(i+1, date)
              monthly_user_actions.insert(i+1, UserActions.ACCOUNT_DETAILS)
              daily_balances.insert(i+1, total_balace)
              main_account_balances.insert(i+1, current_savings_balance)
              main_savings_status.insert(i+1, account_status)
              payment_accounts_from.insert(i+1, "")

              #Avoid duplicating session_id if it was propagated from previous session
              if monthly_session_ids[i-1] == session_id:
                  session_id = generateSessionId()
              monthly_session_ids.insert(i+1, session_id)
              #print("[INFO] session_id **:", session_id)
            #else:
            #  continue
            #Insert TransHistorical action within the next random number of minutes with p probability
            if chance(p=0.45) == True and user_action == UserActions.ACCOUNT_DETAILS:
              print(f"[INFO] changing to trans-hist user_action")
              next_minute = random.randint(date.minute, 59)
              date = date.replace(minute=next_minute)
              #print(f"[INFO] date after summing {next_minute} mins:", date)
              monthly_dates.insert(i+1, date)
              monthly_user_actions.insert(i+1, UserActions.TRANS_HISTORICAL)
              #Avoid duplicating session_id if it was propagated from previous session
              #if monthly_session_ids[i-1] == session_id:
              #    session_id = generateSessionId()
              monthly_session_ids.insert(i+1, session_id)
              daily_balances.insert(i+1, total_balace)
              main_account_balances.insert(i+1, current_savings_balance)
              main_savings_status.insert(i+1, account_status)
              payment_accounts_from.insert(i+1, "")
              #print("[INFO] session_id **:", session_id)

            # Generate next payment action
            (next_action, payment_amount) = getNextPaymentAction(pending_payment_actions=pending_actions, dow=int(date.day), wage_received=first_payment_received)
            #print("[INFO] next_action:", next_action)
            payment_probability = 1.0
            if i>=2:
                previous_first_payment_action = monthly_user_actions[i-1].split("-")
                previous_second_payment_action = monthly_user_actions[i-2].split("-")
                #Damp probability of payment within session if previous paymentes were performed
                if len(previous_first_payment_action) >1:
                    payment_probability *= 0.5
                if len(previous_second_payment_action) >1:
                    payment_probability *= 0.5

            if next_action != "" and chance(p=payment_probability):
            #Append transaction label within next n random minutes with p probability
            #if chance(p=0.45) == True and len(pending_actions)>0:
                next_minute = random.randint(date.minute, 59)
                date = date.replace(minute=next_minute)
                #Update account_from balance and set account_status accordingly
                previous_savings_balance = accounts["SAVINGS"][0].balance
                account_number = accounts["SAVINGS"][0].account_number
                #accounts["SAVINGS"][0].balance-=payment_amount

                current_savings_balance = updateBalance(AccountTypes.SAVINGS, -payment_amount)
                total_balance = getBalances()
                if previous_savings_balance < current_savings_balance:
                    account_status = AccountBalanceStatus.POSITIVE
                elif previous_savings_balance == current_savings_balance:
                    account_status = AccountBalanceStatus.NEUTRAL
                else:
                    account_status = AccountBalanceStatus.NEGATIVE
                #print(f"[INFO] Updated SAVINGS balance: {accounts['SAVINGS'][0].balance}")

                #dates.append(date)
                #TODO get a random payment action and append it to user_actions
                #next_action_type = random.choice(list(pending_actions.keys()))
                #From the picked payment action type, choose also a subtype
                #next_action = random.choice(pending_actions[next_action_type])
                if monthly_session_ids[i+1] != session_id:
                    monthly_dates.insert(i+1, date)
                    monthly_user_actions.insert(i+1, next_action)
                    monthly_session_ids.insert(i+1, session_id)
                    daily_balances.insert(i+1, total_balance)
                    payment_accounts_from.insert(i+1, account_number)
                    main_account_balances.insert(i+1, current_savings_balance)
                    main_savings_status.insert(i+1, account_status)

                else:
                    monthly_dates.insert(i+2, date)
                    monthly_user_actions.insert(i+2, next_action)
                    monthly_session_ids.insert(i+2, session_id)
                    daily_balances.insert(i+2, total_balance)
                    payment_accounts_from.insert(i+2, account_number)
                    main_account_balances.insert(i+2, current_savings_balance)
                    main_savings_status.insert(i+2, account_status)
                ##monthly_dates.insert(i+2, date)
                #monthly_user_actions.insert(i+2, next_action_type+"-"+next_action)
                ##monthly_user_actions.insert(i+2, next_action)
                #if monthly_session_ids[i-1] != session_id:
                #    session_id = monthly_session_ids[i-1]
                #print("[INFO] session_id **:", session_id)
                ##monthly_session_ids.insert(i+2, session_id)
                #Charge transaction to either SAVINGS or CHECKING accounts

                ##daily_balances.insert(i+2, total_balace)
                #print("[INFO] session_id ***:", session_id)
                #user_actions.append(next_action)
                #Remove next_action from pending_actions
                #print("[INFO] next_action:", next_action_type+"-"+next_action)
                print("[INFO] dict before resizing:", pending_actions)
                next_action_type, next_action = next_action.split("-")
                if len(pending_actions[next_action_type]) == 1:
                    pending_actions.pop(next_action_type)
                else:
                    pending_actions[next_action_type].remove(next_action)
                #pending_actions.pop(next_action)
                print("[INFO] dict after resizing:", pending_actions)

                #Normalize session_id
                #if user_action != UserActions.DASHBOARD and monthly_session_ids[i-1] != session_id:


            i+=1

            #print(date)
        dates.extend(monthly_dates)
        user_actions.extend(monthly_user_actions)
        session_ids.extend(monthly_session_ids)
        balances.extend(daily_balances)
        accounts_from.extend(payment_accounts_from)
        main_savings_balance.extend(main_account_balances)
        main_savings_balance_status.extend(main_savings_status)
        #payment_flags.extend(payment_due)
        i=0
        #print(dates)
generateUserActions()

#Sort dates increasingly
#dates.sort()


Streaming output truncated to the last 5000 lines.
[INFO] changing to acc-det user_action
[INFO] returning empty due to no pending payments
[INFO] date: 2025-09-29 02:06:44
[INFO] user_action: Account_Details
[INFO] returning empty due to no pending payments
[INFO] date: 2025-09-29 03:41:53
[INFO] user_action: Dashboard
[INFO] changing to acc-det user_action
[INFO] returning empty due to no pending payments
[INFO] date: 2025-09-29 03:50:53
[INFO] user_action: Account_Details
[INFO] changing to trans-hist user_action
[INFO] returning empty due to no pending payments
[INFO] date: 2025-09-29 03:58:53
[INFO] user_action: Transaction_Historical
[INFO] returning empty due to no pending payments
[INFO] date: 2025-09-29 04:08:48
[INFO] user_action: Dashboard
[INFO] returning empty due to no pending payments
[INFO] date: 2025-09-29 04:30:09
[INFO] user_action: Dashboard
[INFO] returning empty due to no pending payments
[INFO] date: 2025-09-29 07:17:37
[INFO] user_action: Dashboard
[INFO] changi

In [18]:
for date in dates[:100]:
  print(date)

2024-01-01 02:33:51
2024-01-01 02:47:51
2024-01-01 02:52:37
2024-01-01 02:59:37
2024-01-01 06:03:38
2024-01-01 06:52:38
2024-01-01 06:54:38
2024-01-01 07:52:24
2024-01-01 07:58:24
2024-01-01 08:05:34
2024-01-01 09:02:59
2024-01-01 09:48:59
2024-01-01 09:26:53
2024-01-01 09:48:53
2024-01-01 09:41:41
2024-01-01 09:53:09
2024-01-01 09:56:09
2024-01-01 09:56:09
2024-01-01 10:42:11
2024-01-01 10:59:11
2024-01-01 14:10:03
2024-01-01 14:17:03
2024-01-01 14:22:03
2024-01-01 17:33:44
2024-01-01 18:55:20
2024-01-01 18:58:20
2024-01-01 18:59:20
2024-01-01 20:16:49
2024-01-01 20:32:03
2024-01-01 21:24:17
2024-01-01 23:00:45
2024-01-01 23:03:45
2024-01-02 00:13:58
2024-01-02 03:05:29
2024-01-02 03:05:57
2024-01-02 03:47:57
2024-01-02 03:13:30
2024-01-02 03:54:30
2024-01-02 03:55:30
2024-01-02 04:13:41
2024-01-02 04:34:41
2024-01-02 07:03:24
2024-01-02 07:17:24
2024-01-02 08:15:35
2024-01-02 08:34:35
2024-01-02 08:58:35
2024-01-02 08:42:44
2024-01-02 09:41:27
2024-01-02 10:18:45
2024-01-02 10:31:45


In [19]:
for date, action, id, balance, stat in zip(dates, user_actions, session_ids, balances, main_savings_balance_status):
  print(date, action, id, balance, stat)

#Add: sum of total accounts available funds, accountFrom type, amount of payment

2024-01-01 02:33:51 Dashboard 4070a18d-e81b-46f1-8ae5-c0a7f8995441 1247618.3840852454 POSITIVE
2024-01-01 02:47:51 Account_Details 4070a18d-e81b-46f1-8ae5-c0a7f8995441 1247618.3840852454 FLAT
2024-01-01 02:52:37 Dashboard de4b9cb5-38f4-4232-81ff-027efb6acdf4 1247618.3840852454 FLAT
2024-01-01 02:59:37 Account_Details de4b9cb5-38f4-4232-81ff-027efb6acdf4 1247618.3840852454 FLAT
2024-01-01 06:03:38 Dashboard a11d070c-f1e7-4720-8696-37267b545a17 1247618.3840852454 FLAT
2024-01-01 06:52:38 Account_Details a11d070c-f1e7-4720-8696-37267b545a17 1247618.3840852454 FLAT
2024-01-01 06:54:38 PAYMENT_EXPRESS-PURCHASES a11d070c-f1e7-4720-8696-37267b545a17 1240394.3840852454 NEGATIVE
2024-01-01 07:52:24 Dashboard e170d2fc-f1f2-4f2c-9d0b-4c6cf6e9f92a 1240394.3840852454 FLAT
2024-01-01 07:58:24 Account_Details e170d2fc-f1f2-4f2c-9d0b-4c6cf6e9f92a 1240394.3840852454 FLAT
2024-01-01 08:05:34 Dashboard e0c4f542-c3f5-44ea-a7ef-43e7f6c2f559 1240394.3840852454 FLAT
2024-01-01 09:02:59 Dashboard 359f98d7-bca

In [20]:
len(user_actions)

11049

In [21]:
'''
 dates.extend(monthly_dates)
        user_actions.extend(monthly_user_actions)
        session_ids.extend(monthly_session_ids)
        balances.extend(daily_balances)
        accounts_from.extend(payment_accounts_from)
        main_savings_balance.extend(main_account_balances)
        main_savings_balance_status.extend(main_savings_status)
'''

'\n dates.extend(monthly_dates)\n        user_actions.extend(monthly_user_actions)\n        session_ids.extend(monthly_session_ids)\n        balances.extend(daily_balances)\n        accounts_from.extend(payment_accounts_from)\n        main_savings_balance.extend(main_account_balances)\n        main_savings_balance_status.extend(main_savings_status)\n'

In [22]:
for item in accounts_from:
    if item != "":
      print(item)

In [24]:
len(main_savings_balance)

11049

In [26]:
df = pd.DataFrame(list(zip(dates, user_actions, session_ids, balances, accounts_from, main_savings_balance, main_savings_balance_status)),
                  columns=['date', 'user-action', 'session-Id', 'total-balance', 'account-from', 'main-savings-balance', 'main-savings-status'])

In [27]:
df.head(5)

,date,user-action,session-Id,total-balance,account-from,main-savings-balance,main-savings-status
0,2024-01-01 02:33:51,Dashboard,4070a18d-e81b-46f1-8ae5-c0a7f8995441,1.247618e+06,,930644.37,POSITIVE
1,2024-01-01 02:47:51,Account_Details,4070a18d-e81b-46f1-8ae5-c0a7f8995441,1.247618e+06,,930644.37,FLAT
2,2024-01-01 02:52:37,Dashboard,de4b9cb5-38f4-4232-81ff-027efb6acdf4,1.247618e+06,,930644.37,FLAT
3,2024-01-01 02:59:37,Account_Details,de4b9cb5-38f4-4232-81ff-027efb6acdf4,1.247618e+06,,930644.37,FLAT
4,2024-01-01 06:03:38,Dashboard,a11d070c-f1e7-4720-8696-37267b545a17,1.247618e+06,,930644.37,FLAT
